# NUMERIA-Q — teste de acesso ao IBM Quantum

Notebook isolado para testar autenticação, listar QPUs disponíveis e executar um circuito mínimo em hardware real. Não altera as versões congeladas NUMERIA-Q 05.19.1 ou 05.20.

**Atenção:** a célula de execução física envia um trabalho para um QPU e pode consumir sua cota/plano. Não execute até verificar o backend selecionado. Este notebook não usa simulador como fallback.

In [ ]:
%pip -q install 'qiskit>=2.5' 'qiskit-ibm-runtime>=0.47'

import qiskit
import qiskit_ibm_runtime
print('Qiskit:', qiskit.__version__)
print('qiskit-ibm-runtime:', qiskit_ibm_runtime.__version__)

## 1. Autenticação segura

Crie uma conta IBM Quantum Platform, gere uma API key e consulte o CRN da instância na página Instances. O token será solicitado com entrada oculta e não deve ser colocado no código nem enviado nesta conversa.

In [ ]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

IBM_API_KEY = getpass('IBM Quantum API key (entrada oculta): ')
INSTANCE_CRN = input('Instance CRN (copie da página Instances; deixe vazio se não se aplicar): ').strip()
if not IBM_API_KEY.strip():
    raise ValueError('API key vazia; nenhuma conexão iniciada.')

kwargs = {'channel': 'ibm_quantum_platform', 'token': IBM_API_KEY}
if INSTANCE_CRN:
    kwargs['instance'] = INSTANCE_CRN
service = QiskitRuntimeService(**kwargs)
print('Autenticação inicializada. Agora vamos consultar os QPUs visíveis.')
del IBM_API_KEY

In [ ]:
# Apenas consulta; não envia circuito nem cria job.
backends = service.backends(simulator=False, operational=True)
print('QPUs operacionais visíveis:', len(backends))
for backend in backends:
    print({'name': backend.name, 'num_qubits': backend.num_qubits,
           'status': str(backend.status().status_msg),
           'pending_jobs': backend.status().pending_jobs})
if not backends:
    raise RuntimeError('Nenhum QPU operacional visível. Confira acesso, instância, plano e permissões. Nenhum job foi enviado.')

## 2. EXECUTAR NO HARDWARE — teste de um qubit

A célula abaixo seleciona um QPU operacional com menor fila, compila um circuito X+medição para a arquitetura do dispositivo e envia 20 shots. O resultado ideal é 20 uns; zeros podem ocorrer devido a ruído. Confirme que aceita enviar o job antes de executar.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

CONFIRM_HARDWARE_JOB = input('Digite ENVIAR para submeter o teste físico de 20 shots: ').strip()
if CONFIRM_HARDWARE_JOB != 'ENVIAR':
    raise RuntimeError('Envio cancelado pelo usuário; nenhum job submetido.')

backend = service.least_busy(simulator=False, operational=True)
qc = QuantumCircuit(1, 1, name='NUMERIA_Q_one_qubit_X_test')
qc.x(0)
qc.measure(0, 0)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
print('Backend selecionado:', backend.name)
print('Circuito compilado:')
print(isa_circuit.draw())

sampler = SamplerV2(mode=backend)
job = sampler.run([isa_circuit], shots=20)
print('Job ID:', job.job_id())
result = job.result()
counts = result[0].data.c.get_counts()
print('Histograma:', counts)
ones = sum(v for bitstring, v in counts.items() if bitstring.strip() == '1')
zeros = sum(v for bitstring, v in counts.items() if bitstring.strip() == '0')
print(f'1s={ones}; 0s={zeros}; total={ones+zeros}')
print('Job concluído; guardar backend, job ID e histograma no diário de auditoria NUMERIA-Q.')

## 3. Interpretação e auditoria

Este teste verifica autenticação, acesso ao QPU e execução de uma porta X com medição. Não demonstra vantagem quântica nem valida os circuitos de inversão modular do NUMERIA-Q. Registre data/hora, backend, versão dos pacotes, job ID, shots e histograma. Nunca publique a API key ou o CRN se ele for tratado como dado interno da organização.